In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torchvision import datasets, transforms
import time
from sklearn.metrics import classification_report
import numpy as np
import os


In [6]:
# Step 1: Define a simple CNN with Fully Connected Layers

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)  # Fully connected layer before factorization
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.conv2(x)
        x = torch.relu(x)
        x = torch.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

# Step 2: Define a CNN model with low-rank factorization on the Fully Connected Layer

class CNNLowRank(nn.Module):
    def __init__(self, rank=50):
        super(CNNLowRank, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        # Low rank factorization of fully connected layer
        self.fc1_u = nn.Linear(9216, rank, bias=False)
        self.fc1_v = nn.Linear(rank, 128, bias=False)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.conv2(x)
        x = torch.relu(x)
        x = torch.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = self.fc1_u(x)
        x = self.fc1_v(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x


In [7]:


# Step 3: Training function

def train(model, device, train_loader, optimizer, criterion, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}')

# Step 4: Evaluation function for accuracy and classification report

def evaluate(model, device, test_loader):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(target.cpu().numpy())
    
    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy:.2f}%')
    report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(10)])
    return accuracy, report

# Step 5: Inference latency measurement

def measure_latency(model, device, test_loader):
    model.eval()
    latencies = []
    
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            start_time = time.time()
            _ = model(data)
            end_time = time.time()
            latencies.append(end_time - start_time)
            
    avg_latency = np.mean(latencies) / len(data)  # Average latency per sample
    print(f'Average Inference Latency per sample: {avg_latency:.6f} seconds')
    return avg_latency

# Step 6: Measure the model size
def get_model_size(model, model_name):
    torch.save(model.state_dict(), model_name)
    model_size = os.path.getsize(model_name) / 1e6  # Convert to MB
    print(f'Model size: {model_size:.2f} MB')
    return model_size



In [8]:
# Step 7: Define the dataset, transforms, and data loaders

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_set = datasets.FashionMNIST('./data', download=True, train=True, transform=transform)
test_set = datasets.FashionMNIST('./data', download=True, train=False, transform=transform)

train_loader = data.DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = data.DataLoader(test_set, batch_size=64, shuffle=False)

# Step 8: Training and evaluation on two models (original and low-rank)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize two models: original and low-rank factorized
model_original = CNN().to(device)
model_lowrank = CNNLowRank(rank=50).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer_original = optim.Adam(model_original.parameters(), lr=0.001)
optimizer_lowrank = optim.Adam(model_lowrank.parameters(), lr=0.001)

# Train both models
print("Training Original Model")
train(model_original, device, train_loader, optimizer_original, criterion, epochs=5)

print("\nTraining Low-Rank Factorized Model")
train(model_lowrank, device, train_loader, optimizer_lowrank, criterion, epochs=5)

# Step 9: Evaluate both models

print("\nEvaluating Original Model")
accuracy_original, report_original = evaluate(model_original, device, test_loader)

print("\nEvaluating Low-Rank Factorized Model")
accuracy_lowrank, report_lowrank = evaluate(model_lowrank, device, test_loader)

# Step 10: Measure inference latency for both models

print("\nMeasuring Inference Latency for Original Model")
latency_original = measure_latency(model_original, device, test_loader)

print("\nMeasuring Inference Latency for Low-Rank Model")
latency_lowrank = measure_latency(model_lowrank, device, test_loader)

# Step 11: Measure model sizes
model_size_original = get_model_size(model_original, "model_original.pth")
model_size_lowrank = get_model_size(model_lowrank, "model_lowrank.pth")

# Return evaluation metrics
results = {
    "accuracy_original": accuracy_original,
    "accuracy_lowrank": accuracy_lowrank,
    "latency_original": latency_original,
    "latency_lowrank": latency_lowrank,
    "model_size_original": model_size_original,
    "model_size_lowrank": model_size_lowrank,
    "classification_report_original": report_original,
    "classification_report_lowrank": report_lowrank,
}

results


Training Original Model
Epoch 1, Loss: 0.3855
Epoch 2, Loss: 0.2398
Epoch 3, Loss: 0.1806
Epoch 4, Loss: 0.1388
Epoch 5, Loss: 0.1023

Training Low-Rank Factorized Model
Epoch 1, Loss: 0.4047
Epoch 2, Loss: 0.2453
Epoch 3, Loss: 0.1947
Epoch 4, Loss: 0.1558
Epoch 5, Loss: 0.1255

Evaluating Original Model
Accuracy: 92.14%

Evaluating Low-Rank Factorized Model
Accuracy: 92.01%

Measuring Inference Latency for Original Model
Average Inference Latency per sample: 0.000025 seconds

Measuring Inference Latency for Low-Rank Model
Average Inference Latency per sample: 0.000026 seconds
Model size: 4.80 MB
Model size: 1.95 MB


{'accuracy_original': 92.14,
 'accuracy_lowrank': 92.01,
 'latency_original': 2.520934791321967e-05,
 'latency_lowrank': 2.6131131846433993e-05,
 'model_size_original': 4.802836,
 'model_size_lowrank': 1.95252,
 'classification_report_original': '              precision    recall  f1-score   support\n\n           0       0.91      0.84      0.88      1000\n           1       1.00      0.97      0.98      1000\n           2       0.83      0.90      0.87      1000\n           3       0.89      0.95      0.92      1000\n           4       0.90      0.84      0.87      1000\n           5       0.99      0.98      0.98      1000\n           6       0.78      0.80      0.79      1000\n           7       0.96      0.98      0.97      1000\n           8       0.99      0.99      0.99      1000\n           9       0.98      0.96      0.97      1000\n\n    accuracy                           0.92     10000\n   macro avg       0.92      0.92      0.92     10000\nweighted avg       0.92      0.92 